# MILP Selection Analysis

This notebook applies selection criteria, produces the expanded (non-overwriting) dataframe, and generates visualizations.

In [ ]:
import pandas as pd
from railpminer.analysis import apply_selection_criteria, analyze_lp_models
from railpminer.visualization import visualize_single_paper_selected, visualize_paper_matrix_selected

In [ ]:
csv_latest = "experiment_results_metrics_corrected_selected.csv"
fig_path_dir = r"C:\Users\joern\GIT\raiLPminer\67531d7506c81a8c34f5794e\figs"

df = pd.read_csv(csv_latest)
df_processed = analyze_lp_models(df, graph_column='graph')

## Filtering & Selection Criteria

In [ ]:
# Filter for quality models
df_filtered = df_processed[
    (df_processed['corrected_completeness'] != 0) &
    (df_processed['model_coherence'] != 0)
]

# Parameter variation: only Paper_1
df_filtered_parameter = df_filtered[
    df_filtered['paper'].str.contains('1', na=False)
]

# Paper variation: fixed model/temp/workflow across papers
df_filtered_paper = df_filtered[
    (df_filtered['temperature'] == 1.0) &
    (df_filtered['workflow'].str.contains('ZS', na=False)) &
    (df_filtered['model'].str.contains('openai_o4_mini', na=False))
]

criteria_1 = [
    ('minimal_complexity', 'max'),
    ('minimal_complexity', 'min'),
    ('graph_diameter', 'max'),
    ('constraint_variable_ratio', 'max'),
]

## Apply Selection (Parameter Variation)

In [ ]:
df_processed, df_expanded_parameter = apply_selection_criteria(
    df_full=df_processed.copy(),
    df_filtered=df_filtered_parameter,
    group_column='paper',
    criteria=criteria_1,
    result_column_name='analysis_parameter',
)

print(f"df_processed rows with analysis_parameter != 'none': "
      f"{(df_processed['analysis_parameter'] != 'none').sum()}")
print(f"df_expanded_parameter rows (one per criterion): {len(df_expanded_parameter)}")
df_expanded_parameter[['paper', 'model', 'workflow', 'temperature', 'analysis_parameter']]

## Apply Selection (Paper Variation)

In [ ]:
df_processed, df_expanded_paper = apply_selection_criteria(
    df_full=df_processed,
    df_filtered=df_filtered_paper,
    group_column='paper',
    criteria=criteria_1,
    result_column_name='analysis_paper',
)

print(f"df_processed rows with analysis_paper != 'none': "
      f"{(df_processed['analysis_paper'] != 'none').sum()}")
print(f"df_expanded_paper rows (one per criterion): {len(df_expanded_paper)}")
df_expanded_paper[['paper', 'model', 'workflow', 'temperature', 'analysis_paper']]

## Visualize Single Paper (Parameter Variation)

In [ ]:
visualize_single_paper_selected(
    df_processed,
    "Paper_1",
    save_path=f"{fig_path_dir}/single_paper_parameter.pdf",
)

## Visualize Paper Matrix (Paper Variation)

In [ ]:
visualize_paper_matrix_selected(
    df_processed,
    save_path=f"{fig_path_dir}/paper_matrix.pdf",
)